# Jour 3 — Premier run d'entraînement

Ce notebook :
1. Lance un entraînement court (10 000 steps)
2. Charge les CSV de logs
3. Visualise les courbes d'apprentissage

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

## 1. Lancement de l'entraînement (10 000 steps)

In [ ]:
import subprocess
result = subprocess.run(
    [
        'python', '../train.py',
        '--total_steps', '10000',
        '--learning_starts', '500',
        '--batch_size', '128',
        '--run_name', 'day3_run',
        '--print_every', '5',
        '--eval_every', '20',
        '--save_every', '50',
        '--seed', '42',
    ],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

## 2. Chargement des logs

In [ ]:
import os
run_dir = '../runs/day3_run'

ep_df  = pd.read_csv(os.path.join(run_dir, 'episodes.csv'))
upd_df = pd.read_csv(os.path.join(run_dir, 'updates.csv'))

print(f'Épisodes : {len(ep_df)}  |  Updates : {len(upd_df)}')
ep_df.tail()

## 3. Courbes d'apprentissage — épisodes

In [ ]:
def smooth(x, w=10):
    """Moyenne glissante."""
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

def plot_col(ax, col, label, color, ylabel):
    if col not in ep_df.columns:
        ax.set_visible(False)
        return
    raw = ep_df[col].dropna().values
    ax.plot(raw, alpha=0.3, color=color)
    ax.plot(smooth(raw), color=color, linewidth=2, label='moyenne glissante')
    ax.set_ylabel(ylabel)
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plot_col(axes[0,0], 'ep_return',   'Récompense cumulée',        'steelblue',  'Reward')
plot_col(axes[0,1], 'pct_in_safe', '% temps dans plage sûre',   'seagreen',   '%')
plot_col(axes[1,0], 'T_max',       'Température max épisode',   'tomato',     '°C')
plot_col(axes[1,1], 'T_mean',      'Température moyenne',       'darkorange', '°C')
plot_col(axes[2,0], 'ep_length',   'Longueur épisode',          'purple',     'Steps')
plot_col(axes[2,1], 'SoC_final',   'SoC final',                 'teal',       'SoC')

for ax in axes.flat:
    ax.set_xlabel('Épisode')

plt.suptitle('Courbes d\'apprentissage — SAC BatteryThermalEnv', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(run_dir, 'learning_curves.png'), dpi=150)
plt.show()
print('Figure sauvegardée.')

## 4. Métriques SAC (losses, alpha, entropy)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ax, col, label, color in [
    (axes[0,0], 'critic_loss', 'Critic Loss',  'tomato'),
    (axes[0,1], 'actor_loss',  'Actor Loss',   'steelblue'),
    (axes[1,0], 'alpha',       'Alpha (temp)', 'seagreen'),
    (axes[1,1], 'entropy',     'Entropy',      'purple'),
]:
    raw = upd_df[col].dropna().values
    ax.plot(smooth(raw, w=50), color=color, linewidth=1.5)
    ax.set_title(label)
    ax.set_xlabel('Update step')
    ax.grid(alpha=0.3)

plt.suptitle('Métriques SAC', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Épisode de test avec l'agent entraîné

In [ ]:
from envs.battery_thermal_env import BatteryThermalEnv
from agent.sac import SACAgent, SACConfig

env   = BatteryThermalEnv()
agent = SACAgent(obs_dim=5, action_dim=1)
agent.load(os.path.join(run_dir, 'checkpoints', 'sac_final.pt'))

obs, _ = env.reset(seed=99)
done = False
while not done:
    action = agent.select_action(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

T_arr = np.array(env.history['T'])
print(f'Steps: {env._step_count}  |  T_max: {T_arr.max():.1f}°C  |  '
      f'T_mean: {T_arr.mean():.1f}°C  |  '
      f'Reward: {sum(env.history["reward"]):.2f}')

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
steps = range(len(T_arr))

axes[0].plot(steps, T_arr, color='tomato')
axes[0].axhline(env.tc.T_safe_max, color='red',  linestyle='--', linewidth=0.8, label='T_max sûr')
axes[0].axhline(env.tc.T_safe_min, color='blue', linestyle='--', linewidth=0.8, label='T_min sûr')
axes[0].set_ylabel('Température [°C]')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(steps, env.history['SoC'], color='steelblue')
axes[1].set_ylabel('SoC')
axes[1].grid(alpha=0.3)

axes[2].plot(steps, env.history['action'], color='seagreen')
axes[2].set_ylabel('Cooling action')
axes[2].set_xlabel('Step')
axes[2].grid(alpha=0.3)

plt.suptitle('Épisode test — Agent SAC entraîné (déterministe)', fontsize=13)
plt.tight_layout()
plt.show()